In [12]:
import os
import json
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# --------------------------------
# 1. API KEY
# --------------------------------

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# --------------------------------
# 2. INITIALIZE LLM
# --------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# --------------------------------
# 3. LONG-TERM MEMORY
# --------------------------------

memory_file = "user_memory.json"

if os.path.exists(memory_file):
    with open(memory_file, "r") as file:
        user_memory = json.load(file)
else:
    user_memory = {}

# --------------------------------
# 4. CONVERSATION HISTORY
# --------------------------------

chat_history = []

# --------------------------------
# 5. CHAT FUNCTION
# --------------------------------

def chat(user_input):

    global user_memory

    system_prompt = f"""
    You are a professional AI assistant.

    User information stored in memory:
    {json.dumps(user_memory)}

    Instructions:
    1. Remember information shared by the user.
    2. If the user tells you their name, identify it.
    3. Use the user's name when relevant.
    4. Answer naturally and professionally.
    5. Never invent personal information.
    """

    messages = [
        SystemMessage(content=system_prompt)
    ]

    messages.extend(chat_history)

    messages.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(messages)

    # --------------------------------
    # 6. EXTRACT USER INFORMATION
    # --------------------------------

    extraction_prompt = f"""
    Analyze the following user message.

    Current memory:
    {json.dumps(user_memory)}

    User message:
    {user_input}

    Extract any personal information explicitly shared by the user.

    Return ONLY valid JSON.

    Example:
    {{"name": "Dharani"}}

    If no new personal information is shared, return:
    {{}}

    Do not guess or invent information.
    """

    extraction_response = llm.invoke(extraction_prompt)

    try:
        new_memory = json.loads(extraction_response.content)

        # Update memory
        user_memory.update(new_memory)

        # Save permanently
        with open(memory_file, "w") as file:
            json.dump(user_memory, file, indent=4)

    except json.JSONDecodeError:
        print("Memory extraction failed.")

    # --------------------------------
    # 7. SAVE CHAT HISTORY
    # --------------------------------

    chat_history.append(
        HumanMessage(content=user_input)
    )

    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content


# --------------------------------
# 8. INTERACTIVE CHATBOT
# --------------------------------

print("AI Assistant Started!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye!")
        break

    answer = chat(user_input)

    print("\nAI:", answer)
    print()

Enter your Groq API Key: ··········
AI Assistant Started!
Type 'exit' to stop.

You: what is my name

AI: Your name is **Vinotha**.

You: which is my native

AI: I’m not sure I understand the question—are you asking about your native language, your place of origin, or something else? Could you please clarify?

You: okay ill share my native trichy

AI: Thanks for letting me know, **Vinotha**—your native place is Trichy. If you’d like to share anything else (such as your native language or any other details), feel free!

You: now tell the last question

AI: Your previous question was:

**“Which is my native?”**

You: what is the answer that question

AI: The answer to your earlier question “Which is my native?” is **Trichy**.

You: exit
AI: Goodbye!
